# 01 Data, trading rule and descriptive statistics

Owner: Deng Shenbojia (A2, A5).

Data checks, the lagged signal, cumulative returns, the distribution and autocorrelation of d.

All returns are daily **log** returns. Notation (same as `src/strategy.py`):

- $r_t = \ln(C_t / C_{t-1})$: buy-and-hold return on day $t$
- $s_t = 1\{\text{SMA5}_{t-1} > \text{SMA20}_{t-1}\}$: position on day $t$ (1 = hold BTC, 0 = hold USDT), decided with the averages of the **previous** day
- $d_t = s_t r_t - r_t = -(1 - s_t)\, r_t$: excess return of the rule over buy-and-hold, the quantity we test

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats  # skewness, kurtosis, normal Q-Q plot
from statsmodels.tsa.stattools import acf  # sample autocorrelation function
from statsmodels.stats.diagnostic import acorr_ljungbox  # Ljung-Box portmanteau test

from src import config

from src.data import load_daily
from src.strategy import build_series

rng = np.random.default_rng(config.SEED)
config.FIG_DIR.mkdir(parents=True, exist_ok=True)  # outputs/figures/
config.TAB_DIR.mkdir(parents=True, exist_ok=True)  # outputs/tables/
pd.set_option("display.float_format", lambda v: f"{v:.6g}")  # readable small numbers

## 1. Data checks

In [ ]:
raw = load_daily()  # 2,588 daily candles; load_daily() already asserts the checks below
df = build_series(raw)  # 2,557 days of the analysis window (Dec 2018 only warms up the SMAs)

full_days = pd.date_range(raw["date"].min(), raw["date"].max(), freq="D")  # every calendar day we should have
checks = pd.DataFrame(
    [
        ("Rows pulled (incl. Dec 2018 warm-up)", len(raw)),
        ("First candle (UTC date)", raw["date"].min().date()),
        ("Last candle (UTC date)", raw["date"].max().date()),
        ("Missing calendar days", len(full_days.difference(raw["date"]))),  # days absent from the file
        ("Duplicate days", int(raw["date"].duplicated().sum())),
        ("Non-positive closes", int((raw["close"] <= 0).sum())),
        ("Rows in analysis window", len(df)),
        ("Missing values in window (any column)", int(df.isna().sum().sum())),
        ("Candle cut", "00:00 UTC; close at 23:59:59 UTC = 08:00 next day in Singapore"),
    ],
    columns=["check", "result"],
)
checks.to_csv(config.TAB_DIR / "01_data_checks.csv", index=False)  # save the table
checks

All checks pass: 2,588 candles from 2018-12-01 to 2025-12-31 with no missing or duplicate days, and 2,557 complete rows in the analysis window. Binance cuts daily candles at 00:00 UTC, so each "day" closes at 08:00 the next morning Singapore time; the raw file and the exact request URLs are recorded in `data/raw/PULL_LOG.md`.

## 2. Price, moving averages and the lagged signal

Grey bands mark the days the rule holds BTC ($s_t = 1$). Because $s_t$ uses the averages of day $t-1$, a band starts one day **after** SMA5 crosses above SMA20.

In [ ]:
def shade_in_btc(ax, s):
    """Grey bands over the days the rule holds BTC (s = 1)."""
    on = s.astype(bool)  # True on days in BTC
    starts = s.index[on & ~on.shift(1, fill_value=False)]  # first day of each holding spell
    ends = s.index[on & ~on.shift(-1, fill_value=False)]  # last day of each holding spell
    for a, b in zip(starts, ends):  # one band per spell
        ax.axvspan(a, b + pd.Timedelta(days=1), color="0.85", lw=0)  # cover the whole last day


def price_plot(data, title, path, legend="upper left"):
    """Close (log scale) with SMA5 and SMA20; grey = in BTC."""
    fig, ax = plt.subplots(figsize=(10, 4))
    shade_in_btc(ax, data["s"])  # bands first, lines on top
    ax.plot(data.index, data["close"], color="black", lw=0.9, label="Close")
    ax.plot(data.index, data["sma5"], color="C0", lw=0.9, label="SMA5")
    ax.plot(data.index, data["sma20"], color="C3", lw=0.9, label="SMA20")
    ax.set_yscale("log")  # equal % moves look equal
    ax.set_ylabel("BTCUSDT close (log scale)")
    ax.set_title(title)
    ax.legend(loc=legend)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


price_plot(df, "BTCUSDT daily close with SMA5 / SMA20 (grey = rule holds BTC)", config.FIG_DIR / "01_price_sma.png")

The rule is in BTC for 55.4% of days and switches position 143 times in seven years, about once every 18 days.

In [ ]:
price_plot(df.loc["2022"], "2022 zoom: signal uses the SMAs of the previous day", config.FIG_DIR / "01_price_sma_2022.png", legend="upper right")

In 2022 the rule is in BTC only 38% of the time. It exits before the June crash (out of the market on 13 June, when BTC fell 15%), but it is still in BTC during the FTX collapse on 8-9 November (-10% and -14%) and leaves on 10 November, the day BTC rebounded 10%. The one-day lag cuts both ways, and each band starts the day after the crossover, so the signal never uses same-day information.

## 3. Cumulative log returns: rule vs buy-and-hold

In [ ]:
cum = df[["r_rule", "r"]].cumsum()  # cumulative log return = running sum of daily log returns
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cum.index, cum["r"], color="black", lw=1, label=f"Buy and hold (end {cum['r'].iloc[-1]:.2f})")
ax.plot(cum.index, cum["r_rule"], color="C0", lw=1, label=f"MA crossover rule (end {cum['r_rule'].iloc[-1]:.2f})")
ax.axhline(0, color="0.6", lw=0.6)
ax.set_ylabel("Cumulative log return")
ax.set_title("Cumulative log returns, 2019-2025")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(config.FIG_DIR / "01_cumulative.png", dpi=150)
plt.show()
print("Rule:", round(cum["r_rule"].iloc[-1], 3), "| Buy and hold:", round(cum["r"].iloc[-1], 3))

Over 2019-2025 buy-and-hold ends at a cumulative log return of 3.16 and the rule at 2.63. The rule falls behind in five of the seven years (it exits on dips and re-enters late) and gains in 2020 and 2022, so whether the average gap is distinguishable from zero is a question for the tests in notebooks 02 and 03, not something this chart can settle.

## 4. Distribution of d

Since $d_t = -(1 - s_t)\, r_t$, on every day the rule holds BTC its return equals buy-and-hold and $d_t$ is **exactly 0**. That is why 55.4% of the observations sit on a single spike at zero (the same number as the share of days in BTC).

In [ ]:
d = df["d"].to_numpy()  # daily excess log return of the rule over buy-and-hold
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
big = np.abs(d) > 0.2  # 2020-03-12 only (d = +0.50): BTC fell 40% while the rule was in USDT
axes[0].hist(d[~big], bins=120, color="C0", alpha=0.8)  # counts, not density: the spike at 0 is the point
axes[0].annotate(f"not shown: {big.sum()} day at d = {d[big].max():.2f}\n(12 Mar 2020)", xy=(0.97, 0.95),
                 xycoords="axes fraction", ha="right", va="top", fontsize=9)
axes[0].set_xlabel("d (daily excess log return)")
axes[0].set_ylabel("Number of days")
axes[0].set_title("Histogram of d")
stats.probplot(d, dist="norm", plot=axes[1])  # sample quantiles vs normal quantiles (all days)
axes[1].get_lines()[0].set(color="C0", markersize=3)  # data points
axes[1].get_lines()[1].set(color="C3")  # normal reference line
axes[1].set_title("Normal Q-Q plot of d (all days)")
fig.tight_layout()
fig.savefig(config.FIG_DIR / "01_d_distribution.png", dpi=150)
plt.show()

share_zero = np.mean(d == 0)  # share of days with d exactly 0
print("Skewness:", round(stats.skew(d), 2))
print("Excess kurtosis:", round(stats.kurtosis(d, fisher=True), 1))  # 0 for a normal distribution
print("Share of days with d = 0:", round(share_zero, 3), "| share of days in BTC:", round(df["s"].mean(), 3))
print("d = -(1 - s) r holds on every day:", np.allclose(d, -(1 - df["s"]) * df["r"]))
crash = df.loc["2020-03-12", "d"]  # the single largest |d|
print("d on 2020-03-12:", round(crash, 4), "| mean d with / without that day:",
      round(d.mean(), 6), "/", round(df["d"].drop(pd.Timestamp("2020-03-12")).mean(), 6))

$d$ is far from normal: skewness 3.5, excess kurtosis 83 and heavy tails on both sides in the Q-Q plot. One day dominates the right tail: on 12 March 2020 BTC fell 40% (log return -0.50) while the rule was in USDT, giving $d = +0.50$; without that day the mean of $d$ would be about -0.0004 instead of -0.0002. With $n = 2{,}557$ the CLT still applies to the sample mean, but such heavy tails make the normal approximation slower to settle, which is one reason to check the t-test against a resampling method.

## 5. Autocorrelation of d and |d|

Bars outside the grey band are larger than $\pm 1.96/\sqrt{n}$, the approximate 95% range under independence. The Ljung-Box test checks the first 10 lags jointly ($H_0$: no autocorrelation up to lag 10).

In [ ]:
n = len(d)
lags = np.arange(1, 21)  # lags 1-20 days
acf_d = acf(d, nlags=20, fft=True)[1:]  # drop lag 0 (always 1)
acf_abs = acf(np.abs(d), nlags=20, fft=True)[1:]
band = 1.96 / np.sqrt(n)  # approx. 95% band under independence

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, vals, name in [(axes[0], acf_d, "d"), (axes[1], acf_abs, "|d|")]:
    ax.axhspan(-band, band, color="0.85", lw=0, zorder=0)  # inside = consistent with zero autocorrelation
    ax.bar(lags, vals, color="C0", width=0.6, zorder=2)
    ax.set_xticks([1, 5, 10, 15, 20])  # whole days only
    ax.axhline(0, color="black", lw=0.6)
    ax.set_xlabel("Lag (days)")
    ax.set_title(f"ACF of {name}")
axes[0].set_ylabel("Autocorrelation")
fig.tight_layout()
fig.savefig(config.FIG_DIR / "01_acf_d.png", dpi=150)
plt.show()

lb = pd.concat(
    [acorr_ljungbox(d, lags=[10]).assign(series="d"), acorr_ljungbox(np.abs(d), lags=[10]).assign(series="|d|")]
)
lb = lb.rename_axis("lag").reset_index()[["series", "lag", "lb_stat", "lb_pvalue"]]
lb["acf_lag1"] = [acf_d[0], acf_abs[0]]
lb.to_csv(config.TAB_DIR / "01_ljung_box.csv", index=False)
lb

$d$ has a negative first-order autocorrelation of -0.11 and $|d|$ has strong positive autocorrelation (0.36 at lag 1, decaying slowly), and the Ljung-Box test rejects independence for both (p far below 0.001). **The i.i.d. assumption behind the ordinary t-test does not hold, so we resample whole blocks of consecutive days (moving block bootstrap, notebook 03), which keeps this dependence inside each block.** The negative lag-1 autocorrelation of $d$ itself is what affects the variance of $\bar d$ (see Ma's formula in notebook 02), while the clustering in $|d|$ is volatility clustering.

## 6. Summary statistics

In [ ]:
summary = pd.DataFrame(
    {
        "n": [len(df)],
        "mean_d": [df["d"].mean()],
        "sd_d": [df["d"].std(ddof=1)],
        "mean_d_annualised": [df["d"].mean() * 365],  # crypto trades every day
        "share_in_btc": [df["s"].mean()],
        "switches": [int(df["switch"].sum())],
    }
)
summary.to_csv(config.TAB_DIR / "01_summary.csv", index=False)
summary

Over the full sample the mean daily excess return is -0.00021 (about -7.6% a year in log terms) with a daily standard deviation of 0.024, so the mean is less than 1% of one standard deviation.

In [ ]:
by_year = df.groupby(df.index.year).agg(
    n=("d", "size"),
    mean_d=("d", "mean"),
    share_in_btc=("s", "mean"),
    switches=("switch", "sum"),
    buy_hold_return=("r", "sum"),  # log return of BTC that year
    rule_return=("r_rule", "sum"),
)
by_year.index.name = "year"
by_year.to_csv(config.TAB_DIR / "01_d_by_year.csv")
by_year

The sign of $\bar d$ changes from year to year: positive in 2020 and 2022, negative in the other five years. Yearly means are only descriptive here; the pre-registered sub-period tests are in notebook 05.